In [ ]:
# =============================================================================
# FUMD-AI Preprocessing Workflow -- Step 4: Merge SUMO trajectories with the OMNeT++ feature matrix
# =============================================================================
# Step:         4 of 7
# Summary:      Align SUMO/OMNeT++ vehicle ids and merge the SUMO trajectory with the Step 3 feature matrix.
#
# Author(s):
#   - Cristina Bernad (ORCID: 0000-0001-9537-415X)
#   - Sonja Filiposka <sonja.filiposka@finki.ukim.mk> (ORCID: 0000-0003-0034-2855)
#   - Katja Gilly (ORCID: 0000-0002-8985-0639)
#
# Copyright:    (c) 2026 Cristina Bernad, Sonja Filiposka, Katja Gilly
# Repository:   https://github.com/FUMD-AI/fumd-ai-preprocessing-workflow
# Version:      1.1.5
# Funding:      This work has been funded by the FUMD-AI project, an EOSC GRAVITY -
#             Inter Project with Grant Number 25-EOSC-GRV-INTER-013.
#
# -----------------------------------------------------------------------------
# Licence
# Unless otherwise indicated:
#
#   * Source code in this notebook is licensed under the MIT License.
#
#   * Explanatory text and original figures are licensed under Creative
#     Commons Attribution 4.0 International (CC BY 4.0). Input datasets
#     retain the licences stated in their corresponding metadata or
#     source records.
#
# SPDX-License-Identifier: MIT
# -----------------------------------------------------------------------------
#
# Structured, machine-readable metadata for this workflow (authors, license,
# inputs/outputs per step) is also maintained in ro-crate-metadata.json at
# the repository root - update both together if either changes.
# =============================================================================


# Step 4 — Merge SUMO trajectories with the OMNeT++ feature matrix

Part of the **FUMD-AI preprocessing workflow** (step 4 of 7).

**Purpose.** SUMO and OMNeT++ each assign their own vehicle ids, which do not
match each other directly. This notebook uses a SUMO<->OMNeT id mapping file
to align both datasets onto a single id space, then merges the per-timestep
SUMO trajectory (position, speed, lane, ...) with the per-timestep OMNeT++
network-quality matrix produced in Step 3.

**Inputs:**
- `VEHICLES_PATH` - SUMO trajectory CSV (tab-separated), one row per
  vehicle per timestep, including lagged position columns `x-1..x-7` / `y-1..y-7`.
- `OMNET_PATH` - the wide OMNeT++ matrix produced by Step 3.
- `MAPPING_PATH` - SUMO<->VEINS/OMNeT++ id mapping, one line per vehicle,
  formatted as `SUMO=<sumo_id> VEINS=<veins_id>` (e.g. `SUMO=0 VEINS=0`).
  This file is produced directly by the OMNeT++/VEINS simulation itself
  and is treated here as a given input, not something this workflow
  generates.

**Output:** `COMBINED_PATH` - one merged row per `(vehicle, time)` present
in both datasets.


In [ ]:
import pandas as pd


In [ ]:
# ---- Parameters ----
# Defaults chain onto Step 2's/Step 3's default outputs and the bundled
# example mapping file, so this notebook runs out of the box.
VEHICLES_PATH = "sumo_trajectory.csv"          # Step 2 output (input)
OMNET_PATH = "omnet_feature_matrix.csv"        # Step 3 output (input)
MAPPING_PATH = "example-data/sumo_veins_mapping.txt"  # SUMO<->VEINS/OMNeT id mapping (input)
COMBINED_PATH = "combined_dataset.csv"         # merged output -> feeds Step 5's INPUT_PATH


## 1. Load all three inputs

In [ ]:
# SUMO trajectory: one row per (vehicle, timestep), tab-separated
vehicles_data = pd.read_csv(VEHICLES_PATH, delimiter="\t")
# Step 3 output: one row per (vehicle, timestep), tab-separated
omnet_data = pd.read_csv(OMNET_PATH, delimiter="\t")

# id mapping: one "SUMO=<id> VEINS=<id>" line per vehicle, no header row.
# Split each whitespace-separated token on "=" to pull out the two integer
# ids (VEINS is the OMNeT++ vehicular-networking framework, so its id here
# is the same "omnet" id used everywhere else in this workflow).
cars_data = pd.read_csv(MAPPING_PATH, sep=r"\s+", header=None, names=["sumo_tok", "omnet_tok"])
cars_data["sumo"] = cars_data["sumo_tok"].str.split("=").str[1].astype(int)
cars_data["omnet"] = cars_data["omnet_tok"].str.split("=").str[1].astype(int)
cars_data = cars_data[["omnet", "sumo"]]

print("SUMO vehicles:", vehicles_data["veh_id"].nunique())
print("OMNeT vehicles:", omnet_data["Object"].nunique())
print("mapping rows:", len(cars_data))
vehicles_data.head()


## 2. Resolve SUMO<->OMNeT id correspondence over time

A `SUMO=<a> VEINS=<b>` mapping row does not always hold for a vehicle's
*entire* trajectory. Two distinct real-world causes produce a repeated id
in this file:

- **SUMO-side teleportation.** SUMO can "teleport" a vehicle stuck in
  gridlock to a new position on the network rather than removing it.
  VEINS treats the vehicle at its new position as a *new* `car[]` module
  slot, so the same SUMO id gets a new mapping row with a different OMNeT
  id. We treat each teleport hop as a distinct vehicle for labeling
  purposes - its trajectory isn't spatially continuous across the
  teleport, so merging the hops into one vehicle would inject a fake
  instantaneous jump into the migration-labeling logic.
- **OMNeT/VEINS-side module-slot reuse.** VEINS has a fixed pool of
  `car[]` module slots; once one is freed (its vehicle leaves the
  simulation, with no teleport involved), a different, unrelated SUMO
  vehicle can be assigned that same slot later in the run. The mapping
  file then has two rows with the *same* OMNeT id but different SUMO ids.

Either way, the id that repeats is only valid for the time window that
OMNeT++'s own feature matrix (Step 3's output) actually logs it for - not
the vehicle's whole run. An earlier version of this notebook tried to
collapse the mapping into a strict one-to-one table first (keeping the
first OMNeT id seen for a repeated SUMO id) and only then remap ids. That
approach has two problems: it only handles the first cause above (it
silently mis-attributes every teleport hop's trajectory to the *first*
hop's OMNeT id, and Step 4's own merge below then drops the other hops'
feature rows as "unmatched" with no warning), and on a mapping file where
the *second* cause is also present it can raise `ValueError: Replacement
lists must match in length` instead (found on `VoipDl-Urban-900_1`).

Instead, we resolve the correspondence **per timestep**: for each
`(SUMO id, time)` in the trajectory, we pick whichever OMNeT id mapped to
that SUMO id had already started being logged (per Step 3's own `Time`
column) at or before that timestep. This handles both causes above
uniformly, with no assumption that either id is reused at most once.


In [ ]:
# The first Time each OMNeT id appears in Step 3's feature matrix marks
# when that car[] module slot instance - and so that mapping row - became
# active.
omnet_start_time = omnet_data.groupby("Object")["Time"].min().rename("start_time")
mapping_windows = cars_data.merge(omnet_start_time, left_on="omnet", right_index=True, how="left")

n_missing_start = mapping_windows["start_time"].isna().sum()
if n_missing_start:
    print(f"{n_missing_start} mapping row(s) reference an OMNeT id never "
          f"present in the feature matrix - dropping them")
    mapping_windows = mapping_windows.dropna(subset=["start_time"])

n_teleport_hops = mapping_windows["sumo"].duplicated(keep=False).sum()
n_slot_reuses = mapping_windows["omnet"].duplicated(keep=False).sum()
print(f"{n_teleport_hops} mapping row(s) are part of a SUMO id reused across "
      f"multiple OMNeT ids (teleportation)")
print(f"{n_slot_reuses} mapping row(s) are part of an OMNeT id reused across "
      f"multiple SUMO ids (module-slot reuse)")
print("both are resolved per-timestep below, not collapsed to a single id")


## 3. Remap the SUMO trajectory ids into OMNeT id space, per timestep

`pandas.merge_asof` matches each SUMO trajectory row to the most recent
mapping-row start time at or before that row's own timestamp, *within*
that row's own SUMO id (`by=`) - so a teleported vehicle's pre- and
post-teleport rows resolve to different OMNeT ids automatically, without
needing every SUMO id to map to exactly one OMNeT id first.

`merge_asof` requires both frames sorted by the `on` column globally (not
just within each `by` group), so we sort by `t` / `start_time` alone here
- the original row order isn't relied on downstream (the final merge in
part 4 is a `how="right"` join, so its output order already follows
`omnet_data`, not `vehicles_data`).


In [ ]:
vehicles_data = vehicles_data.sort_values("t", kind="stable")
mapping_windows = mapping_windows.sort_values("start_time", kind="stable")

vehicles_data = pd.merge_asof(
    vehicles_data, mapping_windows[["sumo", "omnet", "start_time"]],
    left_on="t", right_on="start_time",
    left_by="veh_id", right_by="sumo",
    direction="backward",
)

n_unresolved = vehicles_data["omnet"].isna().sum()
if n_unresolved:
    print(f"{n_unresolved:,} trajectory row(s) have no OMNeT id yet active "
          f"at their timestamp (e.g. logged just before that vehicle's "
          f"first mapping-file record) - these will be dropped as "
          f"unmatched in the merge below, same as any other observation-"
          f"window mismatch between SUMO and OMNeT++.")

vehicles_data["veh_id"] = vehicles_data["omnet"]
vehicles_data = vehicles_data.drop(columns=["omnet", "sumo", "start_time"])

ids_now_aligned = set(vehicles_data["veh_id"].dropna().unique()) <= set(omnet_data["Object"].unique())
print("every remapped SUMO veh_id has a matching OMNeT Object id:", ids_now_aligned)


## 4. Merge on (time, vehicle id) and save

In [ ]:
# how="right" keeps every OMNeT row (the network-quality signal we care
# about) and attaches the matching SUMO trajectory row where one exists.
combined_data = pd.merge(
    vehicles_data, omnet_data,
    left_on=["t", "veh_id"], right_on=["Time", "Object"],
    how="right",
)
print("rows before dropping unmatched:", len(combined_data))

# rows with no matching SUMO sample (different observation windows between
# the two simulators) end up all-NaN on the SUMO side - drop them
combined_data = combined_data.dropna()
print("rows after dropping unmatched:", len(combined_data))

combined_data.to_csv(COMBINED_PATH, index=False)
print(f"saved {COMBINED_PATH} - shape {combined_data.shape}")
combined_data.head()
